# 05 · Inference and submission files

The full pipeline on the **test** split with the *same code* as training:
`ber.cleaning` → country blocks + TF-IDF top-K → features → LightGBM (from S3) → threshold.

Test pairs (~100M) are scored **in a stream**: features are computed per chunk, scored, and only
(pair, probability) is stored — the full test feature table never exists in memory or on disk.

Outputs in `output/` (matching file also uploaded to S3):
- `matching_results.tsv` — `source1_entity_id \t matched_entity_ids` (the scored file)
- `candidate_pairs.tsv` — `source1_entity_id \t candidate_entity_ids`: exactly the pairs the
  model scored, so every matched ID is also a candidate.

Both have one row per test S1 entity (empty list allowed), unique comma-separated S2/S3 IDs, TAB
separator, UTF-8, and are checked with the official `validate_submission.py`. France (unseen in
training) is processed like any other country block.

In [ ]:
import sys, time, gc
from pathlib import Path

# notebooks/ -> ../src holds the shared, importable pipeline package `ber`
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ber import config as C, io, pipeline as P

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
C.ensure_dirs()
print(C.describe())
try:
    import psutil
    print(f"RAM: {psutil.virtual_memory().total / 1e9:.1f} GB | CPUs: {psutil.cpu_count()}")
except ImportError:
    pass

> **Infrastructure & cost.** Built to run the *full* dataset on a free-tier-class EC2 instance
> (m7i-flex.large: 2 vCPU, 8 GB RAM + 4 GB swap, 30 GB gp3) — every step streams or chunks
> its data. Checkpoints go to the local EBS volume (which survives stop/start) and small,
> valuable artifacts (model, metadata, submission) are mirrored to S3; big intermediates are
> kept local to stay inside the S3 free tier (5 GB, ~2k PUTs/month). S3→EC2 transfer in the
> same region is free. Re-running a notebook skips stages whose checkpoint exists
> (set `FORCE = True` to recompute). Everything is CPU-bound; no GPU is used.

In [ ]:
FORCE = False
if C.SAMPLE_FRAC:
    print(f"WARNING: BER_SAMPLE_FRAC={C.SAMPLE_FRAC} - output will NOT contain every S1 entity and will fail validation.")
import lightgbm as lgb
from ber import model as M
model = lgb.Booster(model_file=str(io.fetch("model/lgbm.txt")))
meta = M.load_meta(io.fetch("model/meta.json"))
print({k: meta.get(k) for k in ("threshold", "one_to_one", "best_iteration", "validB_macro_f05", "validB_precision", "validB_recall")})

## 1. Clean the test sources (identical functions as notebook 00)

In [ ]:
t0 = time.time()
P.run_clean("test", force=FORCE)
print(P.source_sizes("test"), "| countries:", P.countries("test"), f"| {time.time() - t0:.0f}s")

## 2. Candidate generation (same settings as training)

In [ ]:
t0 = time.time()
stats = P.run_candidates("test", force=FORCE)
n1 = P.source_sizes("test")["s1"]
print(f"S1 with zero candidates: {100 * (stats['s1_n'] == 0).mean():.2f}% | pairs {int(stats['r_n'].sum()):,} | {time.time() - t0:.0f}s")
del stats; gc.collect()

## 3. Stream features → scores (resumable per chunk)

In [ ]:
t0 = time.time()
parts = P.run_scoring("test", model, force=FORCE)
print(f"{len(parts)} score parts in {time.time() - t0:.0f}s")

## 4. Apply the decision rule and write the submission files

In [ ]:
t0 = time.time()
summary = P.write_submission("test", meta["threshold"], meta["one_to_one"], C.OUTPUT_DIR)
out_match, out_cand = C.OUTPUT_DIR / "matching_results.tsv", C.OUTPUT_DIR / "candidate_pairs.tsv"
io.s3_upload(out_match, io.s3_key(C.S3_PREFIX, "output", out_match.name), force=True)
print(summary, f"{time.time() - t0:.0f}s")

## 5. Validate with the official checker

In [ ]:
import subprocess
validator = C.REPO_DIR / "scripts" / "validate_submission.py"
res = subprocess.run([sys.executable, str(validator), "--matching", str(out_match), "--candidate", str(out_cand),
                      "--test-dir", str(io.raw_path("test", "s1").parent)], capture_output=True, text=True)
print(res.stdout[-3000:], res.stderr[-2000:])

In [ ]:
pd.read_csv(out_match, sep="\t", dtype=str, keep_default_na=False, nrows=10)